In [21]:
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [23]:
loader=PyPDFLoader("../Data/DT_Gene_Fairoz.pdf")
docs=loader.load()

In [24]:
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=150)
splitted_docs=splitter.split_documents(docs)

In [25]:
embeddings=OllamaEmbeddings(model="nomic-embed-text")
vector_store=InMemoryVectorStore.from_documents(documents=splitted_docs,
                                                embedding=embeddings
                                                )


In [39]:
@tool 
def retreival_tool(query:str):
    """
    This tool should be used to retreive the context for a query.
    """
    retreived_chunks=vector_store.similarity_search(query,k=2)
    print(f'Tool Called ',query)
    context=""
    for data in retreived_chunks:
        context+=data.page_content


    return context

In [40]:
llm=ChatOllama(model="gemma4:31b-cloud")

In [41]:
agent=create_agent(
    model=llm,
    tools=[retreival_tool],
    system_prompt="You are a helpful Assistant that helps with the Q&A  always use the tool  'retreival_tool' for obtaining context and then only answer from the information from the retreival tool"
)

In [42]:
query="What is Fairoz's experience And what are his key acheivements"

In [43]:
res=agent.invoke({"messages":{"role":"user","content":query}})
answer=res['messages'][-1].content

Tool Called  Fairoz experience and key achievements


In [45]:
print(answer)

Fairoz Ahmad Sheikh is an AI Trainer and AI/ML Engineer with the following experience and achievements:

**Experience:**
*   **AI Training & Evaluation:** He has hands-on experience training, evaluating, and improving large language models (LLMs). This includes ranking and correcting model outputs, writing structured critiques and reference solutions, and applying rubric-driven quality standards.
*   **Platforms:** He has worked across multiple AI training platforms, including Invisible Technologies, SuperAnnotate, SME Careers, and Alignerr.
*   **Technical Expertise:** His engineering background includes Python, Machine Learning, Deep Learning, Computer Vision, NLP, and LLM/RAG systems.

**Key Achievements:**
*   **Cybersecurity:** He is ranked among the top cybersecurity learners in India and achieved a Top 0.1% global ranking on TryHackMe.
*   **Coding Consistency:** He maintained a coding streak of over 500 days on GitHub.
